# Minimal Weather ReAct Agent\n
\n
This notebook accepts a place name as input, reasons step-by-step, calls a weather tool, and returns a weather report.\n

In [ ]:
# Minimal Weather ReAct Agent (Google Colab friendly)

import requests

# ---- Tool: weather tool ----
def get_weather(place: str) -> str:
    try:
        # Step 1: Geocode the place name
        geo_url = "https://geocoding-api.open-meteo.com/v1/search"
        geo_params = {
            "name": place,
            "count": 1,
            "language": "en",
            "format": "json"
        }

        geo_response = requests.get(geo_url, params=geo_params, timeout=10)
        geo_response.raise_for_status()
        geo_data = geo_response.json()

        if "results" not in geo_data or not geo_data["results"]:
            return f"weather tool error: Could not find location '{place}'."

        location = geo_data["results"][0]
        name = location.get("name", "Unknown")
        admin1 = location.get("admin1", "")
        country = location.get("country", "")
        latitude = location["latitude"]
        longitude = location["longitude"]

        resolved_location = ", ".join(x for x in [name, admin1, country] if x)

        # Step 2: Fetch current weather
        weather_url = "https://api.open-meteo.com/v1/forecast"
        weather_params = {
            "latitude": latitude,
            "longitude": longitude,
            "current": [
                "temperature_2m",
                "relative_humidity_2m",
                "apparent_temperature",
                "wind_speed_10m",
                "weather_code"
            ]
        }

        weather_response = requests.get(weather_url, params=weather_params, timeout=10)
        weather_response.raise_for_status()
        weather_data = weather_response.json()

        current = weather_data["current"]

        temperature = current["temperature_2m"]
        humidity = current["relative_humidity_2m"]
        feels_like = current["apparent_temperature"]
        wind_speed = current["wind_speed_10m"]
        weather_code = current["weather_code"]

        weather_desc = weather_code_to_text(weather_code)

        return (
            f"Weather report for {resolved_location}:\n"
            f"- Condition: {weather_desc}\n"
            f"- Temperature: {temperature}°C\n"
            f"- Feels like: {feels_like}°C\n"
            f"- Humidity: {humidity}%\n"
            f"- Wind speed: {wind_speed} km/h"
        )

    except Exception as e:
        return f"weather tool error: {e}"


def weather_code_to_text(code: int) -> str:
    mapping = {
        0: "Clear sky",
        1: "Mainly clear",
        2: "Partly cloudy",
        3: "Overcast",
        45: "Fog",
        48: "Depositing rime fog",
        51: "Light drizzle",
        53: "Moderate drizzle",
        55: "Dense drizzle",
        56: "Light freezing drizzle",
        57: "Dense freezing drizzle",
        61: "Slight rain",
        63: "Moderate rain",
        65: "Heavy rain",
        66: "Light freezing rain",
        67: "Heavy freezing rain",
        71: "Slight snow fall",
        73: "Moderate snow fall",
        75: "Heavy snow fall",
        77: "Snow grains",
        80: "Slight rain showers",
        81: "Moderate rain showers",
        82: "Violent rain showers",
        85: "Slight snow showers",
        86: "Heavy snow showers",
        95: "Thunderstorm",
        96: "Thunderstorm with slight hail",
        99: "Thunderstorm with heavy hail",
    }
    return mapping.get(code, f"Unknown weather code ({code})")


TOOLS = {
    "weather": get_weather
}

# ---- Minimal ReAct agent ----
class WeatherReActAgent:
    def __init__(self, tools):
        self.tools = tools

    def reason(self, query: str) -> str:
        return "The user provided a place name. I should call the weather tool and return the report."

    def choose_action(self, query: str):
        place = query.strip()
        if place:
            return ("weather", place)
        return (None, None)

    def run(self, query: str) -> str:
        trace = []
        trace.append(f"Question: {query}")

        thought = self.reason(query)
        trace.append(f"Thought: {thought}")

        action, action_input = self.choose_action(query)

        if action:
            trace.append(f"Action: {action}")
            trace.append(f"Action Input: {action_input}")
            observation = self.tools[action](action_input)
            trace.append(f"Observation: {observation}")
            final = observation
        else:
            final = "No place name was provided."

        trace.append(f"Final Answer: {final}")
        return "\\n".join(trace)

# ---- User prompt ----
agent = WeatherReActAgent(TOOLS)
user_query = input("Enter place name: ")
print(agent.run(user_query))